In [ ]:
import os
os.chdir('D:/Projects/tumor-immune-scrna-atlas')
print(os.getcwd())

import scanpy as sc
import matplotlib.pyplot as plt


from .autonotebook import tqdm as notebook_tqdm

adata = sc.read_h5ad('data/interim/05b_integrated.h5ad')

# Compute UMAP for uncorrected PCA for the comparison panel
sc.pp.neighbors(adata, use_rep='X_pca', key_added='uncorr')
sc.tl.umap(adata, neighbors_key='uncorr')
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()


# Three-method × two-coloring grid
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
 
methods = [
    ('X_umap_uncorrected', 'Uncorrected'),
    ('X_umap_harmony',     'Harmony'),
    ('X_umap_scvi',        'scVI'),
]
 
for col, (key, title) in enumerate(methods):
    adata.obsm['X_umap'] = adata.obsm[key]
    sc.pl.umap(adata, color='sample_id',   ax=axes[0, col],
               show=False, title=f'{title} — by sample',
               legend_loc='right margin', size=3)
    sc.pl.umap(adata, color='CD3E',        ax=axes[1, col],
               show=False, title=f'{title} — CD3E expression',
               cmap='viridis', size=3)
 
plt.tight_layout()
plt.savefig('results/figures/integration_comparison.png', dpi=300)
plt.close()


SyntaxError: invalid syntax (366052155.py, line 1)

In [ ]:
pip install scib-metrics
 
# In the notebook:
from scib_metrics.benchmark import Benchmarker
 
bm = Benchmarker(
    adata,
    batch_key='sample_id',
    label_key='egfr_status',     # biology label - we want this preserved
    embedding_obsm_keys=['X_pca', 'X_pca_harmony', 'X_scVI'],
    n_jobs=4,
)
bm.benchmark()
df = bm.get_results(min_max_scale=False)
df.to_csv('results/tables/integration_metrics.csv')
print(df)

In [ ]:
adata.obsm['X_umap'] = adata.obsm['X_umap_scvi']  # set scVI as default
adata.write('data/interim/05_integrated.h5ad')